# Loading BatMo BDF CSV data

This notebook shows how to load a BatMo BDF CSV file with the built-in `batmo_bdf` loader, inspect the resulting cellpy data object, extract useful pandas DataFrames, make a simple voltage-capacity plot, and export the processed data to other formats.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Changes to the cellpy repo can directly be used without installing the package. This is useful for development and testing.
repo_root = next(
    (path for path in [Path.cwd(), Path.cwd().parent] if (path / "cellpy" / "__init__.py").exists()),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import cellpy

%matplotlib inline

## Locate the example file

The notebook looks for `batmo_bdf.csv` under the repo-root `example_data/data` folder (relative to this notebook: `../../example_data/data`).

In [ ]:
candidates = [
    Path("../../example_data/data/batmo_bdf.csv"),
    Path("example_data/data/batmo_bdf.csv"),
]

raw_file = next((path for path in candidates if path.exists()), None)
if raw_file is None:
    raise FileNotFoundError("Could not find batmo_bdf.csv in example_data/data")

raw_file

## Load with the BatMo loader

BatMo BDF CSV files are loaded by passing `instrument="batmo_bdf"`. The example data starts with a discharge step, so `cycle_mode="anode"` is used here.

In [ ]:
c = cellpy.get(
    raw_file,
    instrument="batmo_bdf",
    cycle_mode="anode",
    mass=1.0,
)

c

## Inspect the processed data

After loading, `cellpy` has generated the raw data table, the step table, and the summary table.

In [ ]:
raw = c.data.raw
steps = c.data.steps
summary = c.data.summary

print(f"Raw points: {len(raw):,}")
print(f"Cycles: {len(c.get_cycle_numbers())}")
print(f"Step types: {steps['type'].value_counts().to_dict()}")

In [ ]:
r = c.schema.raw
raw[
    [
        r.datapoint_num,
        r.test_time,
        r.step_time,
        r.current,
        r.potential,
        r.step_num,
        r.cycle_num,
        r.cumulative_charge_capacity,
        r.cumulative_discharge_capacity,
    ]
].head()

In [ ]:
steps[["cycle", "step", "type", "point_min", "point_max", "voltage_first", "voltage_last"]].head(10)

In [ ]:
summary.head()

## Make a quick raw-data plot

The raw table is a normal pandas DataFrame, so you can use pandas, matplotlib, seaborn, plotly, or the cellpy plotting helpers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw["test_time"] / 3600, raw["voltage"], lw=0.8)
ax.set_xlabel("Test time / h")
ax.set_ylabel("Voltage / V")
ax.set_title("BatMo BDF raw voltage trace")
ax.grid(alpha=0.25);

## Extract voltage-capacity curves

`get_cap()` returns tidy pandas DataFrames that are convenient for plotting or further analysis. Here `mode="absolute"` keeps the capacities in absolute units.

In [ ]:
cycles = [6, 10]
curve = c.get_cap(
    cycles=cycles,
    method="forth-and-forth",
    categorical_column=True,
    label_cycle_number=True,
    mode="absolute",
)

curve.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for (cycle, direction), frame in curve.groupby(["cycle", "direction"]):
    label = f"cycle {cycle} {'charge' if direction > 0 else 'discharge'}"
    ax.plot(frame["capacity"], frame["voltage"], label=label, lw=1.2)

ax.set_xlabel("Capacity / mAh")
ax.set_ylabel("Voltage / V")
ax.set_title("Selected BatMo voltage-capacity curves")
ax.legend(fontsize=8)
ax.grid(alpha=0.25);

## Export to other formats

The processed cellpy object can be saved as a cellpy HDF5 file and exported to CSV or Excel. The CSV export below keeps the output compact by exporting summary and cycle data only.

In [ ]:
out_dir = Path("out/batmo_bdf")
csv_dir = out_dir / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

cellpy_file = out_dir / "batmo_bdf.cellpy"
excel_file = out_dir / "batmo_bdf.xlsx"

c.save(cellpy_file)
c.to_csv(datadir=csv_dir, raw=False, summary=True, cycles=True, last_cycle=5)
c.to_excel(excel_file, cycles=[1, 2, 10], raw=False)

sorted(path.name for path in out_dir.iterdir())

## Reload the saved cellpy file

Once saved as a cellpy file, loading is faster and does not require specifying the BatMo raw-data loader again.

In [ ]:
c2 = cellpy.get(cellpy_file)

print(f"Reloaded raw points: {len(c2.data.raw):,}")
print(f"Reloaded cycles: {len(c2.get_cycle_numbers())}")